# Supplementary figure: 5-FU × Olaparib dose-by-dose similarity, 2D vs 3D

Profile cosine similarity for every concentration pair, with grit per dose strip-annotated on the margins. Two panels: 2D HCT116 (5 × 5) and 3D aggregates HCT116 (4 × 4). The 3D Olaparib 10 µM row uses 2 replicates after dropping the out-of-focus well `PB000138 / O11`.

Reads off the figure:
- In 2D the 5-FU × Olaparib similarity is high across all active doses (≥1 µM × ≥1 µM cosines 0.62–0.89).
- In 3D the matched-dose similarity drops sharply (e.g. 10 µM × 10 µM = 0.48 vs 0.73 in 2D), so the divergence is not a dose-choice artefact.
- Cells where either compound has grit < 1.96 (non-reproducible profile at that dose) are faded — only the opaque cells should be read as biological similarity.

In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, require)
from utils.panels import save_panel

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from cytominer_eval import evaluate

OUT = str(figdir("SupplFig5"))
os.makedirs(OUT, exist_ok=True)

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

GRIT_THRESH = 1.96

In [ ]:
def list_features(df):
    meta = list(df.columns[df.columns.str.contains('Metadata_')])
    return [c for c in df.columns if c not in meta], meta


def run_grit(df):
    """Match the canonical grit pipeline in 3_GritScores.ipynb."""
    df = df.copy()
    df['Metadata_name'] = df['Metadata_cmpdname'].str[:5]
    df['Metadata_conc_step'] = df.groupby('Metadata_cmpdname')['Metadata_cmpd_conc'].rank(
        ascending=True, method='dense')
    df['Metadata_pert_name'] = df['Metadata_name'] + '_' + df['Metadata_cmpd_conc'].astype(str)
    df['Metadata_replicate_id'] = df['Metadata_name'] + '_' + df.index.astype(str)
    control_perts = (
        df.query("Metadata_name == 'dmso' & Metadata_cmpd_conc == 0.1")
          .Metadata_replicate_id.unique().tolist()
    )
    feats, meta = list_features(df)
    res = evaluate(
        profiles=df, features=feats, meta_features=meta,
        replicate_groups={'profile_col': 'Metadata_replicate_id',
                          'replicate_group_col': 'Metadata_pert_name'},
        operation='grit', similarity_metric='pearson',
        grit_replicate_summary_method='median', grit_control_perts=control_perts,
    )
    res['Metadata_name'] = res['perturbation'].str.split('_').str[0]
    res = res.merge(df[['Metadata_replicate_id', 'Metadata_cmpd_conc']],
                    left_on='perturbation', right_on='Metadata_replicate_id')
    return res


def consensus_by_dose(df, name_substr, feats):
    sub = df[df['Metadata_cmpdname'].astype(str).str.contains(
        name_substr, case=False, na=False)]
    return {c: g[feats].median().values for c, g in sub.groupby('Metadata_cmpd_conc')}


def cosine(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))


def similarity_matrix(rows, cols):
    r = sorted(rows); c = sorted(cols)
    M = np.full((len(r), len(c)), np.nan)
    for i, a in enumerate(r):
        for j, b in enumerate(c):
            M[i, j] = cosine(rows[a], cols[b])
    return pd.DataFrame(M, index=r, columns=c)


def grit_per_dose(grit_df, name_5char):
    sub = grit_df[grit_df['Metadata_name'] == name_5char]
    return sub.groupby('Metadata_cmpd_conc')['grit'].mean()

In [ ]:
# 2D
d2 = pd.read_parquet(profiles("exp1_main", "selected_data_2D_HCT116.parquet")).reset_index(drop=True)

# 3D aggregates: drop the out-of-focus Olaparib 10 µM well PB000138 / O11
d3 = pd.read_parquet(profiles("exp1_main", "selected_data_aggregates_HCT116.parquet")).reset_index(drop=True)
drop_mask = (
    d3['Metadata_cmpdname'].astype(str).str.contains('Olapa', case=False)
    & (d3['Metadata_cmpd_conc'] == 10)
    & (d3['Metadata_Barcode'] == 'PB000138')
    & (d3['Metadata_Well'] == 'O11')
)
print(f'Dropping {int(drop_mask.sum())} out-of-focus row from 3D aggregates:')
print(d3.loc[drop_mask, ['Metadata_Barcode', 'Metadata_Well', 'Metadata_cmpdname',
                          'Metadata_cmpd_conc']].to_string())
d3 = d3[~drop_mask].reset_index(drop=True)


def finite_feats_for_pair(df):
    """Features finite across the 5-FU / Olaparib / DMSO wells (the 2D parquet has NaN columns)."""
    feats, _ = list_features(df)
    sub = df[df['Metadata_cmpdname'].astype(str).str.contains(
        'Fluoro|Olapa|DMSO', case=False, na=False, regex=True)]
    return [c for c in feats if sub[c].notna().all()]


feats2 = finite_feats_for_pair(d2)
feats3 = finite_feats_for_pair(d3)
print(f'\n2D features: {len(feats2)} kept')
print(f'3D features: {len(feats3)} kept')

In [ ]:
# Consensus profiles + cosine matrices
fl2 = consensus_by_dose(d2, 'Fluoro', feats2)
ol2 = consensus_by_dose(d2, 'Olapa', feats2)
fl3 = consensus_by_dose(d3, 'Fluoro', feats3)
ol3 = consensus_by_dose(d3, 'Olapa', feats3)

sim2 = similarity_matrix(fl2, ol2)
sim3 = similarity_matrix(fl3, ol3)

print('2D cosine (rows = 5-FU µM, cols = Olaparib µM):')
print(sim2.round(3))
print('\n3D cosine (O11 dropped):')
print(sim3.round(3))

# Grit per dose
grit2 = run_grit(d2)
grit3 = run_grit(d3)
g_fl2 = grit_per_dose(grit2, 'Fluor')
g_ol2 = grit_per_dose(grit2, 'Olapa')
g_fl3 = grit_per_dose(grit3, 'Fluor')
g_ol3 = grit_per_dose(grit3, 'Olapa')

print('\n2D grit (mean per dose):')
print('  5-FU:', g_fl2.round(2).to_dict())
print('  Olap:', g_ol2.round(2).to_dict())
print('3D grit (mean per dose, O11 dropped):')
print('  5-FU:', g_fl3.round(2).to_dict())
print('  Olap:', g_ol3.round(2).to_dict())

In [ ]:
# Shared color scales across both panels
sim_max = max(np.nanmax(np.abs(sim2.values)), np.nanmax(np.abs(sim3.values)))
sim_norm = TwoSlopeNorm(vmin=-sim_max, vcenter=0.0, vmax=sim_max)
cmap_sim = plt.get_cmap('RdBu_r')

grit_vmin = min(g_fl2.min(), g_ol2.min(), g_fl3.min(), g_ol3.min(), 0.0)
grit_vmax = max(g_fl2.max(), g_ol2.max(), g_fl3.max(), g_ol3.max())
grit_norm = plt.Normalize(vmin=grit_vmin, vmax=grit_vmax)
cmap_grit = plt.get_cmap('Blues')  # matches the rest of the paper

WHITE_TEXT_THRESH = 0.7    # |cosine| above which heatmap cell text turns white
LOW_GRIT_ALPHA = 0.18      # alpha for main-heatmap cells where either dose has grit < 1.96

# Publication sizing — total figure width 7"
FS_TITLE = 8     # panel titles
FS_AXLBL = 7     # axis labels
FS_TICK  = 6     # tick labels
FS_CELL  = 6     # numbers inside cells
FS_GRIT  = 6     # numbers inside grit strips
FS_CBLBL = 6.5   # colorbar labels


def confidence_alpha(sim, grit_rows, grit_cols):
    """Alpha matrix: 1.0 where both row and col grit >= threshold, else faded."""
    a = np.full(sim.shape, 1.0)
    row_keys = list(sim.index); col_keys = list(sim.columns)
    for i, r in enumerate(row_keys):
        for j, c in enumerate(col_keys):
            g_r = grit_rows.get(r, np.nan); g_c = grit_cols.get(c, np.nan)
            if np.isnan(g_r) or np.isnan(g_c) or g_r < GRIT_THRESH or g_c < GRIT_THRESH:
                a[i, j] = LOW_GRIT_ALPHA
    return a


def draw_grid(ax, values, cmap, norm, alpha=None, edgecolor='none'):
    """Render an H×W matrix as Rectangle patches (PDF vector path — Adobe-safe).

    Cell (i, j) covers [j-0.5, j+0.5] × [i-0.5, i+0.5] so cell *centers* sit on
    integer coordinates (same convention as imshow with extent), letting us
    reuse the (j, i) text positions unchanged."""
    h, w = values.shape
    for i in range(h):
        for j in range(w):
            v = values[i, j]
            if np.isnan(v):
                continue
            rgba = list(cmap(norm(v)))
            if alpha is not None:
                rgba[3] = float(alpha[i, j])
            ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1.0, 1.0,
                                    facecolor=rgba, edgecolor=edgecolor, linewidth=0))
    ax.set_xlim(-0.5, w - 0.5)
    ax.set_ylim(h - 0.5, -0.5)   # invert y so row 0 is at the top (imshow convention)
    ax.set_aspect('auto')


def draw_panel(fig, slot, sim, grit_rows, grit_cols, title):
    inner = slot.subgridspec(2, 2, width_ratios=[1.8, 10], height_ratios=[1.8, 10],
                              wspace=0.05, hspace=0.05)
    ax_top  = fig.add_subplot(inner[0, 1])
    ax_left = fig.add_subplot(inner[1, 0])
    ax_main = fig.add_subplot(inner[1, 1])
    fig.add_subplot(inner[0, 0]).axis('off')

    rows = list(sim.index); cols = list(sim.columns)
    alpha = confidence_alpha(sim, grit_rows, grit_cols)

    # main heatmap — Rectangle patches
    ax_main.set_facecolor('white')
    draw_grid(ax_main, sim.values, cmap_sim, sim_norm, alpha=alpha)
    ax_main.set_xticks(range(len(cols)))
    ax_main.set_xticklabels([f'{c:g}' for c in cols], fontsize=FS_TICK)
    ax_main.set_yticks(range(len(rows)))
    ax_main.set_yticklabels([f'{c:g}' for c in rows], fontsize=FS_TICK)
    ax_main.set_xlabel('Olaparib (µM)', fontsize=FS_AXLBL)
    ax_main.yaxis.tick_right()
    ax_main.yaxis.set_label_position('right')
    ax_main.set_ylabel('5-FU (µM)', fontsize=FS_AXLBL, rotation=270, labelpad=10)
    ax_main.tick_params(axis='both', length=2, pad=2)
    for i in range(sim.shape[0]):
        for j in range(sim.shape[1]):
            v = sim.values[i, j]
            if np.isnan(v):
                continue
            ax_main.text(j, i, f'{v:.2f}', ha='center', va='center',
                         color='white' if abs(v) > WHITE_TEXT_THRESH else 'black',
                         fontsize=FS_CELL, alpha=alpha[i, j])

    # top strip — Olaparib grit
    top_vals = np.array([grit_cols.get(c, np.nan) for c in cols])[None, :]
    draw_grid(ax_top, top_vals, cmap_grit, grit_norm)
    ax_top.set_xticks([]); ax_top.set_yticks([0])
    ax_top.set_yticklabels(['grit'], fontsize=FS_TICK)
    ax_top.tick_params(axis='both', length=0, pad=2)
    for j, v in enumerate(top_vals[0]):
        if np.isnan(v):
            continue
        star = '*' if v >= GRIT_THRESH else ''
        ax_top.text(j, 0, f'{v:.1f}{star}', ha='center', va='center',
                    color='white' if v > grit_vmax * 0.55 else 'black',
                    fontsize=FS_GRIT)
    ax_top.set_title(title, fontsize=FS_TITLE, pad=4)

    # left strip — 5-FU grit
    left_vals = np.array([grit_rows.get(c, np.nan) for c in rows])[:, None]
    draw_grid(ax_left, left_vals, cmap_grit, grit_norm)
    ax_left.set_yticks([]); ax_left.set_xticks([0])
    ax_left.set_xticklabels(['grit'], fontsize=FS_TICK)
    ax_left.tick_params(axis='both', length=0, pad=2)
    for i, v in enumerate(left_vals[:, 0]):
        if np.isnan(v):
            continue
        star = '*' if v >= GRIT_THRESH else ''
        ax_left.text(0, i, f'{v:.1f}{star}', ha='center', va='center',
                     color='white' if v > grit_vmax * 0.55 else 'black',
                     fontsize=FS_GRIT)


fig = plt.figure(figsize=(7.0, 3.6))
# Two heatmap panels + cosine colorbar on row 0; horizontal grit colorbar
# on row 1 (in the GridSpec, not via fig.add_axes — bbox-safe).
outer = GridSpec(2, 3, width_ratios=[1, 1, 0.04], height_ratios=[1, 0.05],
                 wspace=0.42, hspace=0.55, figure=fig,
                 left=0.04, right=0.95, top=0.88, bottom=0.12)
draw_panel(fig, outer[0, 0], sim2, g_fl2, g_ol2, '2D HCT116')
draw_panel(fig, outer[0, 1], sim3, g_fl3, g_ol3, '3D HCT116 (aggregates)')

# cosine colorbar (vertical, right)
cax = fig.add_subplot(outer[0, 2])
sm_sim = plt.cm.ScalarMappable(cmap=cmap_sim, norm=sim_norm)
cb = fig.colorbar(sm_sim, cax=cax)
cb.set_label('cosine similarity', fontsize=FS_CBLBL)
cb.ax.tick_params(labelsize=FS_TICK, length=2, pad=2)

# grit colorbar (horizontal, bottom — spans the two heatmap columns)
cax2 = fig.add_subplot(outer[1, 0:2])
sm_grit = plt.cm.ScalarMappable(cmap=cmap_grit, norm=grit_norm)
cb2 = fig.colorbar(sm_grit, cax=cax2, orientation='horizontal')
cb2.set_label(f'grit  (* = ≥ {GRIT_THRESH};  faded cells: row or col grit < {GRIT_THRESH})',
              fontsize=FS_CBLBL)
cb2.ax.tick_params(labelsize=FS_TICK, length=2, pad=2)

plt.show()

In [ ]:
out_pdf = f'{OUT}/dose_similarity_5FU_Olaparib_2D_vs_3D.pdf'
out_svg = f'{OUT}/dose_similarity_5FU_Olaparib_2D_vs_3D.svg'
out_png = f'{OUT}/dose_similarity_5FU_Olaparib_2D_vs_3D.png'

# No bbox_inches='tight' — that would override the 7" publication width.
# Heatmap cells are Rectangle patches now, so the PDF/SVG contain no image
# streams (Adobe-safe). SVG is the most editable format for journal touch-ups.
save_panel(fig, 'SupplFig5d',
           data=pd.concat([
               sim2.stack().rename('cosine').reset_index().assign(representation='2D'),
               sim3.stack().rename('cosine').reset_index().assign(representation='3D')],
               ignore_index=True),
           caption='Dose-resolved 5-FU vs olaparib similarity, 2D vs 3D',
           notebook='analysis/3_SupplFigure5/3_dose_similarity_5FU_Olaparib.ipynb')
print('Saved:', os.path.abspath(out_pdf))
print('Saved:', os.path.abspath(out_svg))
print('Saved:', os.path.abspath(out_png))